# 03 - Detect OCN And Publish Candidate Dataset

This notebook loads generations, applies the lexical OCN detector, saves scored rows to Drive, publishes them to Hugging Face, and logs charts to W&B.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import wandb
from datasets import load_dataset

from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe, utc_timestamp
from ocn.detectors import OCNDetector
from ocn.metrics import detection_summary, grouped_ocn_rates, top_patterns

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
_ = login_huggingface("HF_WRITE_ACCESS")
EXPERIMENT_ID = "main_gemma4_qwen35"
DETECTION_RUN_ID = utc_timestamp()
MAIN_GENERATION_REPO = config.get(
    "hf_main_generation_repo",
    f"{config['hf_owner']}/ocn-empty-negations-generations-main-gemma4-qwen35",
)
MAIN_DETECTION_REPO = config.get(
    "hf_main_detection_repo",
    f"{config['hf_owner']}/ocn-empty-negations-detection-main-gemma4-qwen35",
)
detection_config = {
    **config,
    "experiment_id": EXPERIMENT_ID,
    "detection_run_id": DETECTION_RUN_ID,
    "source_repo": MAIN_GENERATION_REPO,
    "output_repo": MAIN_DETECTION_REPO,
}
run = login_wandb(
    project="ocn-empty-negations",
    name=f"detect-{EXPERIMENT_ID}-{DETECTION_RUN_ID}",
    config=detection_config,
)
sns.set_theme(style="whitegrid")

In [ ]:
generations = load_dataset(MAIN_GENERATION_REPO, split="train").to_pandas()
scored = OCNDetector().annotate_rows(generations, text_column="response")
summary = detection_summary(scored)
summary

In [ ]:
scored_path = save_dataframe(
    scored,
    Path(config["drive_data_root"]) / "ocn_detection_main_gemma4_qwen35.csv",
)
repo_url = publish_dataframe_to_hf(
    scored,
    repo_id=MAIN_DETECTION_REPO,
    split="train",
    private=config["hf_private"],
    card_path=REPO_ROOT / "dataset_cards/ocn_detection.md",
    commit_message=f"Publish {EXPERIMENT_ID} detection {DETECTION_RUN_ID}",
)
print("Saved:", scored_path)
print("Published:", repo_url)

In [ ]:
model_rates = grouped_ocn_rates(scored, ["model_id", "model_stage", "decoding"])
category_rates = grouped_ocn_rates(scored, ["category", "variant"])
pattern_counts = top_patterns(scored, 20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=model_rates, y="model_id", x="ocn_rate", hue="decoding", ax=axes[0])
axes[0].set_title("Candidate OCN rate by model")
axes[0].set_xlim(0, 1)
sns.barplot(data=category_rates.head(20), y="category", x="ocn_rate", hue="variant", ax=axes[1])
axes[1].set_title("Top category/variant OCN rates")
axes[1].set_xlim(0, 1)
plt.tight_layout()
fig_path = Path(config["drive_figure_root"]) / "03_ocn_rates_main_gemma4_qwen35.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")

wandb.log({
    **summary.to_dict(),
    "model_rates": wandb.Table(dataframe=model_rates),
    "category_rates": wandb.Table(dataframe=category_rates),
    "top_patterns": wandb.Table(dataframe=pattern_counts),
    "ocn_rate_chart": wandb.Image(str(fig_path)),
})
run.finish()
fig_path